# Premier League 2023/24 predictions

## Importing libraries

In [ ]:
import pandas as pd

# Load the dataset from the excel file
#file_path = 'all-euro-data-2023-2024.xlsx'
#df = pd.read_excel(file_path, sheet_name='E0')

# updating the dataset with the recent games played
df = pd.read_csv('E0.csv')

# Display the first few rows of the dataset
df.head()

## Data cleaning

In [ ]:
# steps:
# 1. Check for missing values
missing_values = df.isnull().sum()

# 2. Check for duplicates
duplicate_rows = df.duplicated().sum()

# Display the results of the checks
missing_values, duplicate_rows

In [ ]:
# Dropping columns with a high number of missing values or irrelevant to our analysis
irrelevant_columns = df.columns[df.isnull().sum() > (0.5 * len(df))]  # Columns with more than 50% missing values
df_cleaned = df.drop(columns=irrelevant_columns)

# Convert 'Date' to datetime
df_cleaned['Date'] = pd.to_datetime(df_cleaned['Date'], format='%d/%m/%Y')

# Fill missing values in other relevant columns with appropriate values (like median or mode)
# For numerical columns, we use median and for categorical, we use mode
for column in df_cleaned.columns:
    if df_cleaned[column].dtype == 'object':
        df_cleaned[column].fillna(df_cleaned[column].mode()[0], inplace=True)
    else:
        df_cleaned[column].fillna(df_cleaned[column].median(), inplace=True)

# Recheck for missing values
remaining_missing_values = df_cleaned.isnull().sum().sum()

# Basic information after cleaning
remaining_missing_values, df_cleaned.info()

df_cleaned.head()


## Feature engineering

### steps:
* Encode Team Names: Use label encoding for 'HomeTeam' and 'AwayTeam'. This will convert team names into numeric values, making them usable for the model.

* Recent Form: Calculate the recent form for each team based on the last 5 matches. We'll use the 'FTR' column to determine wins (W), losses (L), and draws (D). This feature will provide insight into the current performance of the teams.

* Average Goals per Game: Compute the average goals scored per game for both home and away teams. This feature helps understand the offensive strength of the teams.

* Team Points: Calculate the total points accumulated by each team so far in the season. Points are awarded based on wins (3 points), draws (1 point), and losses (0 points).

* Head-to-Head Statistics: Analyze the outcomes of matches between the same pairs of teams earlier in the season.

* Other Statistical Features: Depending on the data available, we can include additional features like average possession, number of shots on target, defensive strength, etc.

## Encode team names

In [ ]:
# Import the LabelEncoder class from the sklearn library
from sklearn.preprocessing import LabelEncoder

# Encode the team names
label_encoder = LabelEncoder()
df_cleaned['HomeTeamEncoded'] = label_encoder.fit_transform(df_cleaned['HomeTeam'])
df_cleaned['AwayTeamEncoded'] = label_encoder.transform(df_cleaned['AwayTeam'])


## Recent form

In [ ]:
# function to calculate recent form
def calculate_form_points(team, df_cleaned, num_matches=5):
    # Filter the matches for the team
    team_matches = df_cleaned[(df_cleaned['HomeTeam'] == team) | (df_cleaned['AwayTeam'] == team)]
    # Sort by date to get the most recent matches
    team_matches = team_matches.sort_values(by='Date', ascending=False)
    # Get the last 'num_matches' matches
    recent_matches = team_matches.head(num_matches)
    # Calculate form points
    form_points = 0
    for _, row in recent_matches.iterrows():
        if row['HomeTeam'] == team and row['FTR'] == 'H':
            form_points += 3
        elif row['AwayTeam'] == team and row['FTR'] == 'A':
            form_points += 3
        elif row['FTR'] == 'D':
            form_points += 1
    return form_points

# Apply the function to get recent form for each team in each match
df_cleaned['HomeTeamRecentForm'] = df_cleaned.apply(lambda x: calculate_form_points(x['HomeTeam'], df_cleaned), axis=1)
df_cleaned['AwayTeamRecentForm'] = df_cleaned.apply(lambda x: calculate_form_points(x['AwayTeam'], df_cleaned), axis=1)

# Average Goals per Game
df_cleaned['HomeTeamAvgGoals'] = df_cleaned.groupby('HomeTeam')['FTHG'].transform('mean')
df_cleaned['AwayTeamAvgGoals'] = df_cleaned.groupby('AwayTeam')['FTAG'].transform('mean')



## Team Points

In [ ]:
# Points will be calculated similar to form but for the whole season
def calculate_team_points(team, df_cleaned):
    team_matches = df_cleaned[(df_cleaned['HomeTeam'] == team) | (df_cleaned['AwayTeam'] == team)]
    points = 0
    for _, row in team_matches.iterrows():
        if row['HomeTeam'] == team and row['FTR'] == 'H':
            points += 3
        elif row['AwayTeam'] == team and row['FTR'] == 'A':
            points += 3
        elif row['FTR'] == 'D':
            points += 1
    return points

df_cleaned['HomeTeamPoints'] = df_cleaned.apply(lambda x: calculate_team_points(x['HomeTeam'], df_cleaned), axis=1)
df_cleaned['AwayTeamPoints'] = df_cleaned.apply(lambda x: calculate_team_points(x['AwayTeam'], df_cleaned), axis=1)


## Head-to-Head Statistics

In [ ]:
# Head-to-Head Statistics
# For simplicity, we can count the number of wins, draws, and losses in matches between the same teams
# Note: This implementation is simplified for demonstration purposes
def head_to_head_stats(home_team, away_team, df_cleaned):
    head_to_head_matches = df_cleaned[(df_cleaned['HomeTeam'] == home_team) & (df['AwayTeam'] == away_team)]
    wins = len(head_to_head_matches[head_to_head_matches['FTR'] == 'H'])
    draws = len(head_to_head_matches[head_to_head_matches['FTR'] == 'D'])
    losses = len(head_to_head_matches[head_to_head_matches['FTR'] == 'A'])
    return wins, draws, losses

# Apply the head-to-head stats function
df_cleaned['HeadToHeadWins'], df_cleaned['HeadToHeadDraws'], df_cleaned['HeadToHeadLosses'] = zip(*df_cleaned.apply(lambda x: head_to_head_stats(x['HomeTeam'], x['AwayTeam'], df_cleaned), axis=1))

In [ ]:
# Calculate Home Team Win Percentage
home_team_wins = df_cleaned[df_cleaned['FTR'] == 'H'].groupby('HomeTeam').size()
total_home_matches = df_cleaned.groupby('HomeTeam').size()
home_team_win_percentage = home_team_wins / total_home_matches
# 
# Calculate Away Team Win Percentage
away_team_wins = df_cleaned[df_cleaned['FTR'] == 'A'].groupby('AwayTeam').size()
total_away_matches = df_cleaned.groupby('AwayTeam').size()
away_team_win_percentage = away_team_wins / total_away_matches
# 
# Create mappings for win percentages
home_team_win_percentage_map = home_team_win_percentage.to_dict()
away_team_win_percentage_map = away_team_win_percentage.to_dict()
# 
# Map the win percentages to the original DataFrame
df_cleaned['HomeTeamWinPercentage'] = df_cleaned['HomeTeam'].map(home_team_win_percentage_map).fillna(0)
df_cleaned['AwayTeamWinPercentage'] = df_cleaned['AwayTeam'].map(away_team_win_percentage_map).fillna(0)
# 
# Check the first few rows to confirm the new features
print(df_cleaned[['HomeTeam', 'HomeTeamWinPercentage', 'AwayTeam', 'AwayTeamWinPercentage']].head())

In [ ]:
df_cleaned.head()

In [ ]:
# get the current position and calculate points for chelsea and the rest of the teams
def calculate_points_goals(row):
    home_points, away_points = 0, 0
    home_goal_diff = row['FTHG'] - row['FTAG']
    away_goal_diff = row['FTAG'] - row['FTHG']

    if row['FTR'] == 'H':
        home_points = 3
    elif row['FTR'] == 'A':
        away_points = 3
    else:
        home_points = away_points = 1

    return pd.Series([home_points, away_points, home_goal_diff, away_goal_diff, row['FTHG'], row['FTAG']])

# List of teams
teams = df_cleaned['HomeTeam'].unique()

# Applying the function to the dataset
df_cleaned[['HomePoints', 'AwayPoints', 'HomeGoalDiff', 'AwayGoalDiff', 'HomeGoals', 'AwayGoals']] = df_cleaned.apply(calculate_points_goals, axis=1)

# Summarizing the data for each team
team_stats = pd.DataFrame(index=teams)

# Calculating total points, goal difference, and goals scored for each team
team_stats['Points'] = df_cleaned.groupby('HomeTeam')['HomePoints'].sum() + df_cleaned.groupby('AwayTeam')['AwayPoints'].sum()
team_stats['GoalDiff'] = df_cleaned.groupby('HomeTeam')['HomeGoalDiff'].sum() + df_cleaned.groupby('AwayTeam')['AwayGoalDiff'].sum()
team_stats['GoalsScored'] = df_cleaned.groupby('HomeTeam')['HomeGoals'].sum() + df_cleaned.groupby('AwayTeam')['AwayGoals'].sum()

# Sorting the teams based on Points, Goal Difference, and Goals Scored
sorted_teams = team_stats.sort_values(by=['Points', 'GoalDiff', 'GoalsScored'], ascending=[False, False, False])

# Finding Chelsea's position
chelsea_position_updated = sorted_teams.index.get_loc('Chelsea') + 1
print('Chelsea is currently in position:', chelsea_position_updated)
sorted_teams.head(), chelsea_position_updated


## Data Preprocessing
### steps:
* Scaling Numeric Features: We'll scale features like recent form, average goals, and points using StandardScaler.
* Handling Categorical Variables: We have already encoded team names. We'll ensure other categorical features, if any, are properly encoded.
* Splitting the Data: We'll split the data into training and test sets. We'll use the training set to train the model and the test set to evaluate the model's performance on unseen data.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# select relevant columns or features
features = ['HomeTeamEncoded', 'AwayTeamEncoded', 'HomeTeamRecentForm', 'AwayTeamRecentForm', 'HomeTeamAvgGoals', 
            'AwayTeamAvgGoals', 'HomeTeamPoints', 'AwayTeamPoints', 'HomeTeamWinPercentage', 'AwayTeamWinPercentage']
target = 'FTR'

# Split the data into training and test sets
X = df_cleaned[features]
y = df_cleaned[target]
y = y.map({'A': 2, 'D': 1, 'H': 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Addressing Class Imbalance
smote = SMOTE(random_state=42, k_neighbors=3)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
X_test_smote, y_test_smote = smote.fit_resample(X_test, y_test)

# Scaling the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_smote)
X_test_scaled = scaler.transform(X_test_smote)

## XGBoost

## Hyperparameter Tuning

In [ ]:
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, train_test_split
from sklearn.metrics import (accuracy_score, f1_score, fbeta_score, recall_score,
                            precision_score, classification_report, roc_auc_score, confusion_matrix, auc)
from hyperopt import fmin, tpe, hp, STATUS_OK
import numpy as np

In [ ]:
# Define the hyperparameter space
space = {
    'n_estimators': hp.quniform('n_estimators', 100, 1000, 50),  # Wide range for exploration
    'max_depth': hp.choice('max_depth', range(3, 15)),      # Flexibility in tree depth
    'learning_rate': hp.loguniform('learning_rate', np.log(0.01), np.log(0.2)), # Fine-tune update speed
    'subsample': hp.uniform('subsample', 0.6, 1),       # Control feature randomness
    'colsample_bytree': hp.uniform('colsample_bytree', 0.6, 1),  # Randomness in tree building
    'gamma': hp.uniform('gamma', 0, 5),                  # Adjust tree splits
    'min_child_weight': hp.quniform('min_child_weight', 1, 20, 1),  # Handle imbalanced data
    'reg_lambda': hp.uniform('reg_lambda', 1e-8, 10),     # Prevent overfitting
}

## Creating the Objective Function so that we can use it in the hyperparameter optimization

In [ ]:
# Objective function with early stopping
def objective(params):
    params.update({key: int(value) if key in ('n_estimators', 'max_depth', 'min_child_weight') else value for key, value in params.items()})
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=101)

    xgb_model = xgb.XGBClassifier(**params, random_state=101, eval_metric=['mlogloss', 'merror'], early_stopping_rounds=100)

    scores = []
    for train_index, test_index in skf.split(X, y):
        xgb_model.fit(X.iloc[train_index], y.iloc[train_index], eval_set=[(X.iloc[test_index], y.iloc[test_index])], verbose=False)
        scores.append(accuracy_score(y.iloc[test_index], xgb_model.predict(X.iloc[test_index])))

    mean_scores = np.mean(scores)
    loss = 1 - mean_scores

    return {'loss': loss, 'status': STATUS_OK, 'params': params}

# Hyperparameter optimization
best = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=100)

# Extract the best parameters
best_params = {key: int(value) if key in ('n_estimators', 'max_depth', 'min_child_weight') else value for key, value
               in best.items() if key not in ('loss', 'status')}
print("\nBest Parameters:", best_params)

In [ ]:
best_params = {'colsample_bytree': 0.9511564416161034, 'gamma': 0.12743005798129836, 
               'learning_rate': 0.19233916708685508, 'max_depth': 1, 'min_child_weight': 19, 
               'n_estimators': 400, 'reg_lambda': 3.661240871859491, 'subsample': 0.7892467821644931}

In [ ]:
# model evaluation using cross validation

xgb_model = xgb.XGBClassifier(**best_params, random_state=101, eval_metric=['mlogloss', 'merror'], 
                              early_stopping_rounds=100, objective='multi:softprob')

# rskf = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=101)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=101)

recall_train, accuracy_train, precision_train, f1_train, confusion_matrices_train = [], [], [], [], []
recall_val, accuracy_val, precision_val, f1_val, confusion_matrices_val = [], [], [], [], []

for fold, (train_index, val_index) in enumerate(skf.split(X, y)):
    X_train, X_val = X.values[train_index], X.values[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]
    
    xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    train_pred = xgb_model.predict(X_train)  # Train set
    y_pred = xgb_model.predict(X_val)     # Validation set

    # Combined metric calculations
    train_metrics = (
        recall_score(y_train, train_pred, average='weighted'), 
        accuracy_score(y_train, train_pred),
        precision_score(y_train, train_pred, average='weighted', zero_division=0),
        f1_score(y_train, train_pred, average='weighted'),
        confusion_matrix(y_train, train_pred)
    )

    val_metrics = (
        recall_score(y_val, y_pred, average='weighted'),
        accuracy_score(y_val, y_pred),
        precision_score(y_val, y_pred, average='weighted', zero_division=0),
        f1_score(y_val, y_pred, average='weighted'),
        confusion_matrix(y_val, y_pred)
    )

    # Append metrics to lists
    recall_train.append(train_metrics[0])
    accuracy_train.append(train_metrics[1])
    precision_train.append(train_metrics[2])
    f1_train.append(train_metrics[3])
    confusion_matrices_train.append(train_metrics[4])

    recall_val.append(val_metrics[0])
    accuracy_val.append(val_metrics[1])
    precision_val.append(val_metrics[2])
    f1_val.append(val_metrics[3])
    confusion_matrices_val.append(val_metrics[4])

# Calculate means of metrics for Training set
mean_recall_train = np.mean(recall_train)
mean_accuracy_train = np.mean(accuracy_train)
mean_precision_train = np.mean(precision_train)
mean_f1_train = np.mean(f1_train)

# Calculate means of metrics for Validation set
mean_recall_val = np.mean(recall_val)
mean_accuracy_val = np.mean(accuracy_val)
mean_precision_val = np.mean(precision_val)
mean_f1_val = np.mean(f1_val)

# Display evaluation metrics for Training set
print("\nTraining Set Metrics:")
print("Mean Recall Train:", mean_recall_train)
print("Mean Accuracy Train:", mean_accuracy_train)
print("Mean Precision Train:", mean_precision_train)
print("Mean F1 Train:", mean_f1_train)

# Display evaluation metrics for Validation set
print("\nValidation Set Metrics:")
print("Mean Recall Val:", mean_recall_val)
print("Mean Accuracy Val:", mean_accuracy_val)
print("Mean Precision Val:", mean_precision_val)
print("Mean F1 Val:", mean_f1_val)

In [ ]:
xgb_model.fit(X_train_scaled, y_train_smote, eval_set=[(X_test_scaled, y_test_smote)], verbose=False)

In [ ]:
# Evaluate the model
y_pred = xgb_model.predict(X_test_scaled)

print('Recall Score:', recall_score(y_test_smote, y_pred, average='weighted'))
print('Accuracy Score:', accuracy_score(y_test_smote, y_pred))
print('Precision Score:', precision_score(y_test_smote, y_pred, average='weighted', zero_division=0))
print('F1 Score:', f1_score(y_test_smote, y_pred, average='weighted'))
print('\nClassification Report:\n', classification_report(y_test_smote, y_pred, zero_division=0))
# confusion_matrix(y_test_smote, y_pred)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='white')

# Get feature importances
feature_importances = xgb_model.feature_importances_
feature_importance_df = pd.DataFrame({'Feature': features, 'Importance': feature_importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df)
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Feature Importance')
plt.show()

In [ ]:
# make predictions for the 2023/24 season
def predict_match(home_team, away_team, final_model, scaler):
    # Create a dataframe with the appropriate format
    # Encode home team
    home_team_encoded = df_cleaned.loc[df_cleaned['HomeTeam'] == home_team, 'HomeTeamEncoded'].values[0]
    # Encode away team
    away_team_encoded = df_cleaned.loc[df_cleaned['HomeTeam'] == away_team, 'HomeTeamEncoded'].values[0]

    # Calculate recent form for home and away teams
    home_recent_form = calculate_form_points(home_team, df_cleaned)
    away_recent_form = calculate_form_points(away_team, df_cleaned)
    
    match = pd.DataFrame({
        'HomeTeamEncoded': home_team_encoded,
        'AwayTeamEncoded': away_team_encoded,
        'HomeTeamRecentForm': [home_recent_form],
        'AwayTeamRecentForm': [away_recent_form],
        'HomeTeamAvgGoals': df_cleaned[df_cleaned['HomeTeam'] == home_team]['HomeTeamAvgGoals'].values[0],
        'AwayTeamAvgGoals': df_cleaned[df_cleaned['AwayTeam'] == away_team]['AwayTeamAvgGoals'].values[0],
        'HomeTeamPoints': [calculate_team_points(home_team, df_cleaned)],
        'AwayTeamPoints': [calculate_team_points(away_team, df_cleaned)],
        'HomeTeamWinPercentage': home_team_win_percentage[home_team],
        'AwayTeamWinPercentage': away_team_win_percentage[away_team],
    })
    
    # Scale the features
    match_scaled = scaler.transform(match)

    # Make predictions for the specific match (prediction probabilities)
    match_prediction_proba = xgb_model.predict_proba(match_scaled)

    # Display only the probabilities of interest
    home_team_win_prob = match_prediction_proba[0, 0]
    away_team_win_prob = match_prediction_proba[0, 2]
    draw_prob = match_prediction_proba[0, 1]

    print("Prediction Probabilities for the match between {} (Home Team) and {} (Away Team):\n".format(home_team, away_team))
    print(f"{home_team} win probability: {home_team_win_prob}")
    print(f"{away_team} win probability: {away_team_win_prob}")
    print(f"Draw probability: {draw_prob}", "\n")

In [ ]:
# newcastle vs nott'm forest
predict_match('Newcastle', "Nott'm Forest", xgb_model, scaler)

# bournemouth vs fulham
predict_match('Bournemouth', 'Fulham', xgb_model, scaler)

# sheffield vs luton
predict_match('Sheffield United', 'Luton', xgb_model, scaler)

# burnley vs liverpool
predict_match('Burnley', 'Liverpool', xgb_model, scaler)

# manchester utd vs aston villa
predict_match('Man United', 'Aston Villa', xgb_model, scaler)

## CatBoost

In [ ]:
import catboost as cb
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, recall_score,
                            precision_score, classification_report, roc_auc_score, confusion_matrix, auc)
from hyperopt import fmin, tpe, hp, STATUS_OK
import random

In [ ]:
# hyperparams
space = {
    'iterations': hp.quniform('iterations', 100, 1000, 50),  # CatBoost equivalent of n_estimators
    'depth': hp.choice('depth', range(3, 15)),  # Tree depth
    'learning_rate': hp.loguniform('learning_rate', -6, -1),  # Update speed
    'random_strength': hp.uniform('random_strength', 0.6, 1),  # Feature randomness (subsample + colsample_bytree)
    'bagging_temperature': hp.uniform('bagging_temperature', 0, 5),  # Tree diversity (similar to gamma)
    'min_data_in_leaf': hp.quniform('min_data_in_leaf', 1, 20, 1),  # Handle imbalanced data (min_child_weight equivalent)
    'l2_leaf_reg': hp.uniform('l2_leaf_reg', 1e-8, 100),   # Regularization (reg_lambda equivalent)
}

# Objective function
def objective(params):
    # Convert certain hyperparameters to integers
    params.update({key: int(value) if key in ('iterations', 'depth', 'min_data_in_leaf') else value for key, value in params.items()})
    # skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=101)
    rskf = RepeatedStratifiedKFold(n_splits=6, n_repeats=3, random_state=101)

    cb_model = cb.CatBoostClassifier(**params, random_state=101, eval_metric='Accuracy', loss_function='MultiClass', verbose=False)

    scores = [accuracy_score(y.iloc[test_index], cb_model.fit(X.iloc[train_index], y.iloc[train_index])
                       .predict(X.iloc[test_index]))
              for train_index, test_index in rskf.split(X, y)]

    mean_scores = np.mean(scores)
    loss = 1 - mean_scores

    return {'loss': loss, 'status': STATUS_OK, 'params': params}

# Hyperparameter optimization
best = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=100)

# Extract the best parameters
best_params = {key: int(value) if key in ('iterations', 'depth', 'min_data_in_leaf') else value
               for key, value in best.items() if key not in ('loss', 'status')}

print("\nBest Parameters:", best_params)

In [ ]:
best_params = {'bagging_temperature': 3.0090809959120604, 'depth': 4, 'iterations': 500, 'l2_leaf_reg': 53.92384537926865, 
               'learning_rate': 0.003479923214209124, 'min_data_in_leaf': 14, 'random_strength': 0.7877915886582812}

In [ ]:
# best_params = {
#         'iterations': 700,
#         'depth': 5,
#         'learning_rate': 0.05,
#         'random_strength': 0.6040744959424611,
#         'bagging_temperature': 1.9393475978353383,
#         'min_data_in_leaf': 3,
#         # 'l2_leaf_reg': 5.0
# }

# best_params = {
#         'bagging_temperature': 0.5879791268434761,
#         'depth': 5,
#         'iterations': 800,
#         'l2_leaf_reg': 5,
#         'learning_rate': 0.0012485635588485247,
#         'min_data_in_leaf': 5,
#         'random_strength': 0.6725010905169013
# }

In [ ]:
cb_model = cb.CatBoostClassifier(**best_params, random_state=101, eval_metric='MultiClass', 
                                 loss_function='MultiClass', early_stopping_rounds=200)

# rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=101)
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=101)

recall_train, accuracy_train, precision_train, f1_train, confusion_matrices_train = [], [], [], [], []
recall_val, accuracy_val, precision_val, f1_val, confusion_matrices_val = [], [], [], [], []

for fold, (train_index, val_index) in enumerate(skf.split(X, y)):
    X_train, X_val = X.values[train_index], X.values[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    cb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    train_pred = cb_model.predict(X_train)  # Train set
    y_pred = cb_model.predict(X_val)     # Validation set

    # Combined metric calculations
    train_metrics = (
        recall_score(y_train, train_pred, average='weighted'), 
        accuracy_score(y_train, train_pred),
        precision_score(y_train, train_pred, average='weighted', zero_division=0),
        f1_score(y_train, train_pred, average='weighted'),
        confusion_matrix(y_train, train_pred)
    )

    val_metrics = (
        recall_score(y_val, y_pred, average='weighted'),
        accuracy_score(y_val, y_pred),
        precision_score(y_val, y_pred, average='weighted', zero_division=0),
        f1_score(y_val, y_pred, average='weighted'),
        confusion_matrix(y_val, y_pred)
    )

    # Append metrics to lists
    recall_train.append(train_metrics[0])
    accuracy_train.append(train_metrics[1])
    precision_train.append(train_metrics[2])
    f1_train.append(train_metrics[3])
    confusion_matrices_train.append(train_metrics[4])

    recall_val.append(val_metrics[0])
    accuracy_val.append(val_metrics[1])
    precision_val.append(val_metrics[2])
    f1_val.append(val_metrics[3])
    confusion_matrices_val.append(val_metrics[4])

# Calculate means of metrics for Training set
mean_recall_train = np.mean(recall_train)
mean_accuracy_train = np.mean(accuracy_train)
mean_precision_train = np.mean(precision_train)
mean_f1_train = np.mean(f1_train)

# Calculate means of metrics for Validation set
mean_recall_val = np.mean(recall_val)
mean_accuracy_val = np.mean(accuracy_val)
mean_precision_val = np.mean(precision_val)
mean_f1_val = np.mean(f1_val)

# Display evaluation metrics for Training set
print("\nTraining Set Metrics:")
print("Mean Recall Train:", mean_recall_train)
print("Mean Accuracy Train:", mean_accuracy_train)
print("Mean Precision Train:", mean_precision_train)
print("Mean F1 Train:", mean_f1_train)

# Display evaluation metrics for Validation set
print("\nValidation Set Metrics:")
print("Mean Recall Val:", mean_recall_val)
print("Mean Accuracy Val:", mean_accuracy_val)
print("Mean Precision Val:", mean_precision_val)
print("Mean F1 Val:", mean_f1_val)

In [ ]:
cb_model.fit(X_train_scaled, y_train_smote, eval_set=[(X_test_scaled, y_test_smote)], verbose=False)

In [ ]:
# Evaluate the model
y_pred = cb_model.predict(X_test_scaled)

print('Recall Score:', recall_score(y_test_smote, y_pred, average='weighted'))
print('Accuracy Score:', accuracy_score(y_test_smote, y_pred))
print('Precision Score:', precision_score(y_test_smote, y_pred, average='weighted', zero_division=0))
print('F1 Score:', f1_score(y_test_smote, y_pred, average='weighted'))
print('\nClassification Report:\n', classification_report(y_test_smote, y_pred, zero_division=0))
# confusion_matrix(y_test_smote, y_pred)

## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GridSearchCV

# Addressing Class Imbalance
smote = SMOTE(random_state=42, k_neighbors=3)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
X_test_smote, y_test_smote = smote.fit_resample(X_test, y_test)

# Scaling the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_smote)
X_test_scaled = scaler.transform(X_test_smote)

In [ ]:
# Define the hyperparameter space
space = {
    'n_estimators': hp.quniform('n_estimators', 100, 1000, 50),
    'max_depth': hp.choice('max_depth', range(3, 15)),
    'min_samples_split': hp.uniform('min_samples_split', 0.1, 1.0),
    'max_features': hp.choice('max_features', ['sqrt', 'log2', None]),
    'min_samples_leaf': hp.uniform('min_samples_leaf', 0.1, 0.5),
    'bootstrap': hp.choice('bootstrap', [True, False]),
    'criterion': hp.choice('criterion', ['gini', 'entropy']),
}

In [ ]:
# Objective function with early stopping for Random Forest
def objective(params):
    params['n_estimators'] = int(params['n_estimators'])
    params['max_depth'] = int(params['max_depth'])

    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=101)
    rf_model = RandomForestClassifier(**params, random_state=101)

    scores = []
    for train_index, test_index in skf.split(X, y):
        rf_model.fit(X.iloc[train_index], y.iloc[train_index])
        scores.append(accuracy_score(y.iloc[test_index], rf_model.predict(X.iloc[test_index])))

    mean_scores = np.mean(scores)
    loss = 1 - mean_scores

    return {'loss': loss, 'status': STATUS_OK, 'params': params}

# Hyperparameter optimization for Random Forest
best_rf = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=100)

# Extract the best parameters
best_params_rf = {key: int(value) if key in ('n_estimators', 'max_depth') else value for key, value
                  in best_rf.items() if key not in ('loss', 'status')}
print("\nBest Parameters for Random Forest:", best_params_rf)

In [ ]:
# best_params_rf = {'bootstrap': True, 'criterion': 'gini', 'max_depth': 3, 'max_features': 1, 'min_samples_leaf': 0.16546505884892385, 
#                   'min_samples_split': 0.33036023906633016, 'n_estimators': 150}

In [ ]:
best_params_rf = {'bootstrap': True, 'criterion': 'entropy', 'max_depth': 10, 'max_features': 'sqrt', 
                  'min_samples_leaf': 0.16546505884892385, 'min_samples_split': 0.33036023906633016, 'n_estimators': 800}

# Initialize the Random Forest model with the best hyperparameters
rf_model = RandomForestClassifier(**best_params_rf, random_state=101, n_jobs=-1)

# Stratified K-Fold cross-validation
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=101)

recall_train, accuracy_train, precision_train, f1_train, confusion_matrices_train = [], [], [], [], []
recall_val, accuracy_val, precision_val, f1_val, confusion_matrices_val = [], [], [], [], []

for fold, (train_index, val_index) in enumerate(skf.split(X, y)):
    X_train, X_val = X.values[train_index], X.values[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]
    
    rf_model.fit(X_train, y_train)
    train_pred = rf_model.predict(X_train)  # Train set
    y_pred = rf_model.predict(X_val)     # Validation set

    # Combined metric calculations
    train_metrics = (
        recall_score(y_train, train_pred, average='weighted'), 
        accuracy_score(y_train, train_pred),
        precision_score(y_train, train_pred, average='weighted', zero_division=0),
        f1_score(y_train, train_pred, average='weighted'),
        confusion_matrix(y_train, train_pred)
    )

    val_metrics = (
        recall_score(y_val, y_pred, average='weighted'),
        accuracy_score(y_val, y_pred),
        precision_score(y_val, y_pred, average='weighted', zero_division=0),
        f1_score(y_val, y_pred, average='weighted'),
        confusion_matrix(y_val, y_pred)
    )

    # Append metrics to lists
    recall_train.append(train_metrics[0])
    accuracy_train.append(train_metrics[1])
    precision_train.append(train_metrics[2])
    f1_train.append(train_metrics[3])
    confusion_matrices_train.append(train_metrics[4])

    recall_val.append(val_metrics[0])
    accuracy_val.append(val_metrics[1])
    precision_val.append(val_metrics[2])
    f1_val.append(val_metrics[3])
    confusion_matrices_val.append(val_metrics[4])

# Calculate means of metrics for Training set
mean_recall_train = np.mean(recall_train)
mean_accuracy_train = np.mean(accuracy_train)
mean_precision_train = np.mean(precision_train)
mean_f1_train = np.mean(f1_train)

# Calculate means of metrics for Validation set
mean_recall_val = np.mean(recall_val)
mean_accuracy_val = np.mean(accuracy_val)
mean_precision_val = np.mean(precision_val)
mean_f1_val = np.mean(f1_val)

# Display evaluation metrics for Training set
print("\nTraining Set Metrics:")
print("Mean Recall Train:", mean_recall_train)
print("Mean Accuracy Train:", mean_accuracy_train)
print("Mean Precision Train:", mean_precision_train)
print("Mean F1 Train:", mean_f1_train)

# Display evaluation metrics for Validation set
print("\nValidation Set Metrics:")
print("Mean Recall Val:", mean_recall_val)
print("Mean Accuracy Val:", mean_accuracy_val)
print("Mean Precision Val:", mean_precision_val)
print("Mean F1 Val:", mean_f1_val)

In [ ]:
rf_model.fit(X_train_scaled, y_train_smote)

# Evaluate the model
y_pred = rf_model.predict(X_test_scaled)

print('Recall Score:', recall_score(y_test_smote, y_pred, average='weighted'))
print('Accuracy Score:', accuracy_score(y_test_smote, y_pred))
print('Precision Score:', precision_score(y_test_smote, y_pred, average='weighted', zero_division=0))
print('F1 Score:', f1_score(y_test_smote, y_pred, average='weighted'))
print('\nClassification Report:\n', classification_report(y_test_smote, y_pred, zero_division=0))
# confusion_matrix(y_test_smote, y_pred)

In [ ]:
# Hyperparameter Tuning
model = RandomForestClassifier(random_state=42)
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
    
}
grid_search = GridSearchCV(model, param_grid, cv=3, scoring='f1_macro')
grid_search.fit(X_train_scaled, y_train_smote)

# Best Model Evaluation
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_scaled)
print(classification_report(y_test_smote, y_pred))

In [ ]:
# Evaluate the model
y_pred = best_model.predict(X_test_scaled)

print('Recall Score:', recall_score(y_test_smote, y_pred, average='weighted'))
print('Accuracy Score:', accuracy_score(y_test_smote, y_pred))
print('Precision Score:', precision_score(y_test_smote, y_pred, average='weighted', zero_division=0))
print('F1 Score:', f1_score(y_test_smote, y_pred, average='weighted'))
print('\nClassification Report:\n', classification_report(y_test_smote, y_pred, zero_division=0))
# confusion_matrix(y_test_smote, y_pred)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from scipy.stats import randint

# Hyperparameter Tuning with Randomized Search
model = RandomForestClassifier(random_state=101)

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=101)
random_search = GridSearchCV(model, param_grid, cv=skf, scoring='accuracy')
random_search.fit(X_train_scaled, y_train_smote)

# Best Model Evaluation
best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test_scaled)

print('Recall Score:', recall_score(y_test_smote, y_pred, average='weighted'))
print('Accuracy Score:', accuracy_score(y_test_smote, y_pred))
print('Precision Score:', precision_score(y_test_smote, y_pred, average='weighted', zero_division=0))
print('F1 Score:', f1_score(y_test_smote, y_pred, average='weighted'))
print('\nClassification Report:\n', classification_report(y_test_smote, y_pred, zero_division=0))

In [ ]:
# Save the best model using joblib
model_filename = 'rf_model2.joblib'
dump(best_model, model_filename)

In [ ]:
best_model

In [ ]:
# check for overfitting
from sklearn.model_selection import cross_val_score

# Perform cross-validation
cv_scores = cross_val_score(best_model, X_train_scaled, y_train_smote, cv=6)

# Print the average score
print(f"Average Cross-Validation Score: {cv_scores.mean()}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='white')

# Get feature importances
feature_importances = best_model.feature_importances_
feature_importance_rf = pd.DataFrame({'Feature': features, 'Importance': feature_importances})
feature_importance_rf = feature_importance_rf.sort_values(by='Importance', ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_rf)
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Feature Importance')
plt.show()

In [ ]:
import warnings;warnings.filterwarnings('ignore')

In [ ]:
# make predictions for the 2023/24 season
def predict_match(home_team, away_team, final_model, scaler):
    # Create a dataframe with the appropriate format
    # Encode home team
    home_team_encoded = df_cleaned.loc[df_cleaned['HomeTeam'] == home_team, 'HomeTeamEncoded'].values[0]
    # Encode away team
    away_team_encoded = df_cleaned.loc[df_cleaned['HomeTeam'] == away_team, 'HomeTeamEncoded'].values[0]

    # Calculate recent form for home and away teams
    home_recent_form = calculate_form_points(home_team, df_cleaned)
    away_recent_form = calculate_form_points(away_team, df_cleaned)
    
    match = pd.DataFrame({
        'HomeTeamEncoded': home_team_encoded,
        'AwayTeamEncoded': away_team_encoded,
        'HomeTeamRecentForm': [home_recent_form],
        'AwayTeamRecentForm': [away_recent_form],
        'HomeTeamAvgGoals': df_cleaned[df_cleaned['HomeTeam'] == home_team]['HomeTeamAvgGoals'].values[0],
        'AwayTeamAvgGoals': df_cleaned[df_cleaned['AwayTeam'] == away_team]['AwayTeamAvgGoals'].values[0],
        'HomeTeamPoints': [calculate_team_points(home_team, df_cleaned)],
        'AwayTeamPoints': [calculate_team_points(away_team, df_cleaned)],
        'HomeTeamWinPercentage': home_team_win_percentage[home_team],
        'AwayTeamWinPercentage': away_team_win_percentage[away_team],
    })
    
    # Scale the features
    match_scaled = scaler.transform(match)

    # Make predictions for the specific match (prediction probabilities)
    match_prediction_proba = best_model.predict_proba(match_scaled)

    # Display only the probabilities of interest
    home_team_win_prob = match_prediction_proba[0, 0]
    away_team_win_prob = match_prediction_proba[0, 2]
    draw_prob = match_prediction_proba[0, 1]

    print("Prediction Probabilities for the match between {} (Home Team) and {} (Away Team):\n".format(home_team, away_team))
    print(f"{home_team} win probability: {home_team_win_prob}")
    print(f"{away_team} win probability: {away_team_win_prob}")
    print(f"Draw probability: {draw_prob}", "\n")

In [ ]:
# newcastle vs nott'm forest
predict_match('Newcastle', "Nott'm Forest", best_model, scaler)

# bournemouth vs fulham
predict_match('Bournemouth', 'Fulham', best_model, scaler)

# sheffield vs luton
predict_match('Sheffield United', 'Luton', best_model, scaler)

# burnley vs liverpool
predict_match('Burnley', 'Liverpool', best_model, scaler)

# manchester utd vs aston villa
predict_match('Man United', 'Aston Villa', best_model, scaler)

In [ ]:
# make predictions for the 2023/24 season

matches = [('Crystal Palace', "Brighton"),
           ('West Ham', 'Man United'),
           ('Fulham', 'Burnley'),
           ('Burnley', 'Liverpool'),
           ('Luton', 'Newcastle'),
           ("Nott'm Forest", "Bournemouth"),
           ("Tottenham", "Everton"),
           ("Liverpool", "Arsenal"),
          ("Wolves", "Chelsea")]

for home_team, away_team in matches:
    predict_match(home_team, away_team, best_model, scaler)

In [ ]:
# save random forest (best model) model
import pickle
model_filename = 'rf_model.pkl'

# Save the trained model
with open(model_filename, 'wb') as model_file:
    pickle.dump(best_model, model_file)

print(f"Best Random Forest model saved to {model_filename}")

# # File path of the saved model
# model_filename = 'rf_model.pkl'

# # Load the trained model
# with open(model_filename, 'rb') as model_file:
#     loaded_rf_model = pickle.load(model_file)

In [ ]:
from joblib import dump, load

# Save the best model using joblib
model_filename = 'rf_model.joblib'
dump(best_model, model_filename)
print(f"Best Random Forest model saved to {model_filename}")

# Load the trained model
loaded_model = load(model_filename)

In [ ]:
loaded_model.predict(X_test_scaled)

In [ ]:
# compare the two models
from sklearn.metrics import classification_report

# Evaluate the model
y_pred = final_model.predict(X_test_scaled)
print(classification_report(y_test, y_pred, zero_division=0))

# Evaluate the model
y_pred = best_model.predict(X_test_scaled)
print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:
# plot the model comparison
import matplotlib.pyplot as plt
import seaborn as sns

# Create a DataFrame with the results
model_comparison = pd.DataFrame({'model': ['CatBoost', 'Random Forest'], 'accuracy': [0.59, 0.62]})
# Plot the results
plt.figure(figsize=(8, 6))

sns.barplot(x='model', y='accuracy', data=model_comparison)
plt.ylim(0, 1)
plt.title('Model Comparison')
plt.show()



In [ ]:
predict_match('Wolves', 'Chelsea', best_model, scaler)


In [ ]:
# save the model
import pickle

pickle.dump(best_model, open('best_model.pkl', 'wb'))
pickle.dump(scaler, open('scaler.pkl', 'wb'))
pickle.dump(label_encoder, open('label_encoder.pkl', 'wb'))



In [ ]:
# load the model
import pickle

best_model = pickle.load(open('best_model.pkl', 'rb'))
scaler = pickle.load(open('scaler.pkl', 'rb'))
label_encoder = pickle.load(open('label_encoder.pkl', 'rb'))

